<a href="https://colab.research.google.com/github/jessicawaters/DATA-315-2019-2025-SCIAC-Softball-Web-Scraping/blob/main/DATA_315_2019_2025_SCIAC_Softball.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## SCIAC Softball

In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

def scrape_team(team_name, team_url, years):
    all_games = []

    for year in years:
        url = f"https://{team_url}.com/sports/softball/schedule/{year}"
        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")

        games = soup.find_all("li", class_="sidearm-schedule-game")

        for g in games:
            result_div = g.find("div", class_="sidearm-schedule-game-result")
            if result_div:
                spans = result_div.find_all("span")

                if len(spans) >= 3:
                    outcome = spans[1].text.strip().replace(",", "")
                    score = spans[2].text.strip()

                    score_for = int(re.search(r'\d+', score.split('-')[0]).group())
                    score_against = int(re.search(r'\d+', score.split('-')[1]).group())

                    score_diff = score_for - score_against
                    winner = team_name if outcome == "W" else "Opponent"
                    total_points = score_for + score_against
                    margin_type = "close" if abs(score_diff) <= 2 else ("blowout" if abs(score_diff) >= 5 else "moderate")

                    opp_div = g.find("div", class_="sidearm-schedule-game-opponent-name")
                    opponent = opp_div.get_text(strip=True) if opp_div else ""

                    date_div = g.find("div", class_="sidearm-schedule-game-opponent-date")
                    date_text = date_div.get_text(" ", strip=True) if date_div else ""

                    all_games.append({
                        "team": team_name,
                        "year": year,
                        "opponent": opponent,
                        "date": date_text,
                        "outcome": outcome,
                        "score": score,
                        "score_for": score_for,
                        "score_against": score_against,
                        "score_diff": score_diff,
                        "winner": winner,
                        "total_points": total_points,
                        "margin_type": margin_type
                    })

    return pd.DataFrame(all_games)

In [9]:
# Define Years
years = [2019, 2020, 2021, 2022, 2023, 2024, 2025]

In [13]:
# Scrape each team website
df_clu = scrape_team("California Lutheran", "clusports", years)

df_redlands = scrape_team("Redlands", "goredlands", years)

df_chapman = scrape_team("Chapman", "chapmanathletics", years)

df_pomona = scrape_team("Pomona-Pitzer", "sagehens", years)

df_whittier = scrape_team("Whittier", "wcpoets", years)

df_oxy = scrape_team("Occidental", "oxyathletics", years)

df_cms = scrape_team("Claremont-Mudd-Scripps", "cmsathletics", years)

df_laverne = scrape_team("La Verne", "leopardathletics", years)

In [14]:
# Combine all the teams into one data set
df_final = pd.concat([
    df_clu,
    df_redlands,
    df_chapman,
    df_pomona,
    df_whittier,
    df_oxy,
    df_cms,
    df_laverne
], ignore_index=True)

df_final.shape

(1633, 12)

In [16]:
df_final

,team,year,opponent,date,outcome,score,score_for,score_against,score_diff,winner,total_points,margin_type
0,California Lutheran,2019,La Verne,Feb 16 (Sat) 12 pm,L,4-11,4,11,-7,Opponent,15,blowout
1,California Lutheran,2019,La Verne,Feb 16 (Sat) 2 pm,L,4-6,4,6,-2,Opponent,10,close
2,California Lutheran,2019,Claremont-Mudd-Scripps,Feb 23 (Sat) 12 pm,L,6-9,6,9,-3,Opponent,15,moderate
3,California Lutheran,2019,Claremont-Mudd-Scripps,Feb 23 (Sat) 2 pm,L,0-10,0,10,-10,Opponent,10,blowout
4,California Lutheran,2019,Hiram,Mar 4 (Mon) 12 pm,W,5-1,5,1,4,California Lutheran,6,moderate
...,...,...,...,...,...,...,...,...,...,...,...,...
1628,La Verne,2025,University of Redlands(DH),Apr 26 (Sat) 12 PM,L,3-11,3,11,-8,Opponent,14,blowout
1629,La Verne,2025,University of Redlands(DH),Apr 26 (Sat) 2 PM,L,1-6,1,6,-5,Opponent,7,blowout
1630,La Verne,2025,Whittier College,May 2 (Fri) 3 PM,W,3-0,3,0,3,La Verne,3,moderate
1631,La Verne,2025,Whittier College(DH),May 3 (Sat) 12 PM,L,0-8,0,8,-8,Opponent,8,blowout


In [17]:
# Save to SQL
import sqlite3

conn = sqlite3.connect("sciac_softball.db")
df_final.to_sql("games", conn, if_exists="replace", index=False)
conn.close()

## Final Observations


In [18]:
df_final.groupby("team")["score_diff"].mean()

,score_diff
team,
California Lutheran,-1.701923
Chapman,0.704724
La Verne,0.022727
Occidental,-3.330317
Pomona-Pitzer,1.004184
Redlands,1.674419
Whittier,0.377682


oof

In [19]:
# Win percentages by team
df_final["win_flag"] = df_final["outcome"].apply(lambda x: 1 if x == "W" else 0)

df_final.groupby("team")["win_flag"].mean().sort_values(ascending=False)

,win_flag
team,
Redlands,0.647287
Chapman,0.590551
Pomona-Pitzer,0.543933
Whittier,0.523605
La Verne,0.486364
California Lutheran,0.375000
Occidental,0.212670


In [20]:
# Average points scored vs allowed
df_final.groupby("team")[["score_for", "score_against"]].mean()

,score_for,score_against
team,,
California Lutheran,4.350962,6.052885
Chapman,4.598425,3.893701
La Verne,4.545455,4.522727
Occidental,3.307692,6.638009
Pomona-Pitzer,5.100418,4.096234
Redlands,5.864341,4.189922
Whittier,5.218884,4.841202


In [22]:
# Trends over time
df_final.groupby("year")["score_diff"].mean()

,score_diff
year,
2019,-0.589655
2020,-1.474227
2021,-0.559701
2022,-0.134276
2023,0.161172
2024,0.066176
2025,0.771127


In [24]:
# Win % by team per year
df_final.groupby(["team", "year"])["win_flag"].mean()

team                 year
California Lutheran  2019    0.300000
                     2020    0.416667
                     2021    0.166667
                     2022    0.297297
                     2023    0.411765
                     2024    0.441176
                     2025    0.545455
Chapman              2019    0.452381
                     2020    0.375000
                     2021    0.434783
                     2022    0.666667
                     2023    0.627907
                     2024    0.613636
                     2025    0.750000
La Verne             2019    0.425000
                     2020    0.687500
                     2021    0.375000
                     2022    0.619048
                     2023    0.552632
                     2024    0.351351
                     2025    0.410256
Occidental           2019    0.128205
                     2020    0.076923
                     2021    0.157895
                     2022    0.275000
                     2023    0.250000
                     2024    0.194444
                     2025    0.289474
Pomona-Pitzer        2019    0.704545
                     2020    0.363636
                     2021    0.391304
                     2022    0.488372
                     2023    0.560976
                     2024    0.473684
                     2025    0.615385
Redlands             2019    0.400000
                     2020    0.733333
                     2021    0.666667
                     2022    0.650000
                     2023    0.613636
                     2024    0.702128
                     2025    0.784314
Whittier             2019    0.533333
                     2020    0.428571
                     2021    0.590909
                     2022    0.564103
                     2023    0.540541
                     2024    0.500000
                     2025    0.475000
Name: win_flag, dtype: float64

## Writing Portion

### Data Source / What I Did

I took data from the SCIAC softball conference for the years 2019-2025. This data includes the game results, scores, and opponent information. I then parsed through it and tried to pull some interesting statistics that I thought were meaningful.

After cleaning and organizing the dataset, I explored patterns across teams. I found that some teams had a consistently higher average score difference, indicating stronger overall performance. Additionally, win percentage provided a clear measure of team success, while comparing points scored and allowed revealed whether teams relied more on offense or defense.

### Who is interested?

I believe coaches and players would find this work interesting. If I was able to pull hitting/pitching statistics from each game I think this would be really helpful to analysts.

### Challenges

I defintetly had some challenges trying to get over 1000 observations, which led me to scraping the entire SCIAC conference data. Additionally, I had wanted to parse through the NCAA websites for all games but I could not look through the HTML code.